# Nik Studio - video model test

One picture, one clip. Nothing here touches your project.

This is a **test, not the tool**. It answers one question before any code
gets written: can a free Colab GPU animate your character well enough to
be worth building on?

**Runtime > Change runtime type > T4 GPU** first, then run the cells in
order. Cell 3 takes 10-20 minutes.


In [ ]:
# ======================================================================
# CELL 1 - the packages, and check we really have a GPU
# ======================================================================
#
# Runtime > Change runtime type > T4 GPU  must be set BEFORE running this.

!pip install -q "diffusers>=0.31" transformers accelerate safetensors imageio-ffmpeg sentencepiece

import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Runtime > Change runtime type > T4 GPU, then run again."
    )

name = torch.cuda.get_device_name(0)
total = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU: {name}  ({total:.1f}GB)")

# A T4 cannot do bfloat16, which is what this model prefers. float16 works
# but can wash out or streak. If that happens it is the card, not the code.
BF16 = torch.cuda.is_bf16_supported()

DTYPE = torch.bfloat16 if BF16 else torch.float16

print("Using", "bfloat16 (good)" if BF16 else "float16 (T4 - may show artefacts)")


In [ ]:
# ======================================================================
# CELL 2 - the picture to animate, and what should happen in it
# ======================================================================

from pathlib import Path

from PIL import Image

# The model was trained at this size. Anything else composes badly.
WIDTH, HEIGHT = 720, 480

# --------------------------------------------------------------- picture

# Looked for in Drive first. Nik Studio moves finished images off Drive and
# onto your PC, so there is usually nothing there - the upload box below is
# the normal way in. Pick Images\Scene01.png from your episode folder.

def find_in_drive():

    root = Path("/content/drive/MyDrive")

    if not root.exists():
        return None

    for pattern in ("NikStudio/**/Scene01.*", "NikStudio/**/Images/*.png"):
        for found in sorted(root.glob(pattern)):
            if found.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp"):
                return found

    return None


try:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

except ImportError:
    pass

SOURCE = find_in_drive()

if SOURCE:
    print("Found in Drive:", SOURCE)
else:
    print("Nothing found in Drive - upload one image (Images\\Scene01.png).")

    from google.colab import files

    uploaded = files.upload()

    SOURCE = Path("/content") / next(iter(uploaded))

# The picture is 16:9 and the model wants 3:2, so cover and crop rather
# than squash - a stretched child is not a fair test of the model.
picture = Image.open(SOURCE).convert("RGB")

scale = max(WIDTH / picture.width, HEIGHT / picture.height)

picture = picture.resize(
    (round(picture.width * scale), round(picture.height * scale)),
    Image.LANCZOS,
)

left = (picture.width - WIDTH) // 2
top = (picture.height - HEIGHT) // 2

IMAGE = picture.crop((left, top, left + WIDTH, top + HEIGHT))

display(IMAGE)

# ---------------------------------------------------------------- prompt

# This describes the MOVEMENT, not the picture. The picture is already
# there. Say what moves, and keep it to one clear action - asking for
# three things at once gets you none of them.

PROMPT = (
    "The little boy claps his hands and bounces happily up and down, "
    "smiling and laughing. Soap bubbles drift slowly through the air "
    "around him. The camera holds steady. Pixar style 3D animation, "
    "smooth natural motion, cheerful."
)

# 49 frames at 8fps is 6 seconds and is what the model was trained on.
# 25 frames (3 seconds) is roughly half the wait for this first test.
FRAMES = 25
STEPS = 30
FPS = 8

print("\n", PROMPT)


In [ ]:
# ======================================================================
# CELL 3 - make the clip.  This is the slow one: roughly 10-20 minutes
# ======================================================================
#
# A free T4 does not have the memory to hold this model, so it is streamed
# through the card a piece at a time. That is why it is slow. It is also
# why it fits at all - without it the run dies with "CUDA out of memory".

import gc
import time

import torch

from diffusers import CogVideoXImageToVideoPipeline

# Anything left over from a previous run keeps the card full.
for leftover in ("pipe",):
    if leftover in globals():
        del globals()[leftover]

gc.collect()
torch.cuda.empty_cache()

started = time.time()

pipe = CogVideoXImageToVideoPipeline.from_pretrained(
    "zai-org/CogVideoX-5b-I2V",
    torch_dtype=DTYPE,
)

pipe.enable_sequential_cpu_offload()
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()

print(f"Model ready in {time.time() - started:.0f}s. Generating...")

started = time.time()

frames = pipe(
    image=IMAGE,
    prompt=PROMPT,
    num_frames=FRAMES,
    num_inference_steps=STEPS,
    guidance_scale=6.0,
    generator=torch.Generator("cpu").manual_seed(42),
).frames[0]

print(f"Done in {(time.time() - started) / 60:.1f} minutes.")


In [ ]:
# ======================================================================
# CELL 4 - watch it, and keep a copy
# ======================================================================

from pathlib import Path

from diffusers.utils import export_to_video

OUT = Path("/content/drive/MyDrive/NikStudio/VideoTest")

if not OUT.parent.exists():
    OUT = Path("/content")     # no Drive - keep it beside the notebook

OUT.mkdir(parents=True, exist_ok=True)

clip = OUT / "Scene01_test.mp4"

export_to_video(frames, str(clip), fps=FPS)

print("Saved:", clip)

from IPython.display import Video, display

display(Video(str(clip), embed=True, width=720))

# ----------------------------------------------------------------------
# Now look at it and answer one question: does the child MOVE, and does
# he still look like your character?
#
#   Yes  -> tell Claude "clip achha hai" and the video backend gets built.
#   Face changed / melted -> reference strength problem, fixable.
#   Barely moves -> the prompt needs a stronger action verb.
#   Washed out or streaky -> that is float16 on a T4, not the prompt.
# ----------------------------------------------------------------------
